# V-JEPA 2 physics probing: quick start

Runs on a free Colab GPU (Runtime → Change runtime type → GPU). Upload `vjepa-physics-probe-kit.zip` to the session first (Files panel), then run the cells in order.

In [ ]:
!unzip -q -o vjepa-physics-probe-kit.zip && cd vjepa-physics-probe-kit && pip -q install -e .
%cd vjepa-physics-probe-kit

## 1. Check the analysis code (no weights needed, about a minute)

In [ ]:
!python -m physprobe selftest

## 2. Stimuli: the paper's 392-video velocity set

In [ ]:
!python -m physprobe make velocity --out data/velocity
from IPython.display import Image, display
display(Image('data/velocity/preview.png'))

## 3. Run V-JEPA 2-L and cache activations
Downloads `facebook/vjepa2-vitl-fpc64-256` from Hugging Face on first use.

In [ ]:
!python -m physprobe extract --data data/velocity --out runs/vjepa2-l/velocity/acts --model vjepa2-l --res 224 \
    --fc1-layers all --patch-layers thirds --time-layers thirds

## 4. Where do speed and direction become decodable? (paper Fig. 2)

In [ ]:
!python -m physprobe probe --acts runs/vjepa2-l/velocity/acts --out runs/vjepa2-l/velocity/probe
display(Image('runs/vjepa2-l/velocity/probe/layer_curves.png'))

## 5. What shape is the direction code?

In [ ]:
!python -m physprobe manifold --acts runs/vjepa2-l/velocity/acts --out runs/vjepa2-l/manifold --by direction_deg
import glob
for f in sorted(glob.glob('runs/vjepa2-l/manifold/*.png')): display(Image(f))

## 6. Steering (paper Sec. 7.2) with the low-dimensional control

In [ ]:
!python -m physprobe steer --acts runs/vjepa2-l/velocity/acts --out runs/vjepa2-l/steer --strict
display(Image('runs/vjepa2-l/steer/steering_L8.png'))

## 7. Everything else
`bash scripts/run_replication.sh` runs the full paper pipeline; `bash scripts/run_manifold_zoo.sh` runs the extra manifold experiments. Both end with a `report.html`.

In [ ]:
!bash scripts/run_replication.sh

## Working with activations directly

In [ ]:
from physprobe.extract import ActivationCache
from physprobe import manifold as mf
c = ActivationCache('runs/vjepa2-l/velocity/acts')
X = c.layer(8)                      # [392 videos, 1024] residual stream after block 8, mean-pooled
keys, means, n = mf.condition_means(X, c.labels, ['direction_deg'])
P = mf.pca(means, 3)
print(P['var_ratio'], mf.ring_fit(P['scores'][:, :2], keys.direction_deg.to_numpy(float)))